# DETOURA — 100% Free Cloud Backend Deployment
### Powered by Google Colab & Cloudflare Quick Tunnel (Zero Cost Hosting)

This notebook deploys the existing **Detoura ML Backend** (FastAPI + Gemma 2B GGUF + SQLite + Cloud Firestore) on Google Colab's free runtime and exposes it to the public internet via Cloudflare Quick Tunnel (`trycloudflare.com`).

---
### Architecture Overview
1. **Host Environment:** Google Colab Free Tier (12.7 GB RAM, ~100 GB Disk, CPU / T4 GPU).
2. **AI Model:** Gemma 2B Instruction-Tuned GGUF (`gemma-2-2b-it-Q4_K_M.gguf`, ~1.71 GB).
3. **Database:** Embedded SQLite catalog (`detoura_master-v2.db`) + Google Cloud Firestore.
4. **Networking:** Cloudflare Quick Tunnel creates an instant public HTTPS endpoint (`https://*.trycloudflare.com`) with zero signup, zero credit card, and zero costs.
5. **Frontend:** Production Vercel web app at `https://detoura.vercel.app`.


## Cell 1: Environment & Resource Inspection
Inspect system RAM, disk space, Python runtime, and GPU availability.

In [ ]:
import sys
import os
import shutil
import psutil

print("==================================================")
print("  DETOURA BACKEND — ENVIRONMENT INSPECTION")
print("==================================================")
print(f"Python Version      : {sys.version.split()[0]}")
print(f"Working Directory   : {os.getcwd()}")

# Memory Inspection
ram = psutil.virtual_memory()
print(f"System RAM Total    : {ram.total / (1024**3):.2f} GiB")
print(f"System RAM Available: {ram.available / (1024**3):.2f} GiB")

# Disk Inspection
disk = shutil.disk_usage('/')
print(f"Disk Space Total    : {disk.total / (1024**3):.2f} GiB")
print(f"Disk Space Free     : {disk.free / (1024**3):.2f} GiB")

# GPU Inspection
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU Acceleration    : ENABLED ({torch.cuda.get_device_name(0)})")
        print(f"GPU VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")
    else:
        print("GPU Acceleration    : Not active (Operating in high-efficiency multi-threaded CPU mode)")
except Exception:
    print("GPU Acceleration    : Standard CPU Runtime")

# Compatibility check for Gemma 2B (requires >= 2.5 GB RAM and >= 3 GB disk)
if ram.available / (1024**3) >= 2.5 and disk.free / (1024**3) >= 3.0:
    print("\n[COMPATIBILITY VERIFIED] Environment meets hardware requirements for Gemma 2B GGUF inference.")
else:
    print("\n[WARNING] Low memory or disk. System will prioritize deterministic high-fidelity fallback mode.")


## Cell 2: Obtain Deployment Files
Extract the clean Detoura backend package into `/content/Detoura`.

In [ ]:
import os
import zipfile
from pathlib import Path

TARGET_DIR = Path('/content/Detoura')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(TARGET_DIR))

# Check if detoura-backend.zip already uploaded, otherwise prompt upload
zip_candidates = [
    Path('/content/detoura-backend.zip'),
    Path('/content/Detoura/detoura-backend.zip'),
    Path('detoura-backend.zip')
]
existing_zip = next((p for p in zip_candidates if p.exists()), None)

if not existing_zip:
    print("[ACTION REQUIRED] Please upload detoura-backend.zip from your local D:\\Detoura\\deployment folder:")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                existing_zip = Path(f'/content/{fname}')
                break
    except Exception as e:
        print(f"Upload widget notice: {e}")

if existing_zip and existing_zip.exists():
    print(f"\nExtracting {existing_zip} into {TARGET_DIR}...")
    with zipfile.ZipFile(existing_zip, 'r') as zf:
        zf.extractall(TARGET_DIR)
    print("Extraction complete. Verified files:")
    for item in sorted(TARGET_DIR.glob('*')):
        if item.is_file():
            print(f"  - {item.name} ({item.stat().st_size:,} bytes)")
        elif item.is_dir():
            print(f"  - {item.name}/ ({len(list(item.glob('*')))} files)")
else:
    print("[ERROR] detoura-backend.zip not found. Please upload it to /content/ and rerun this cell.")


## Cell 3: Install Backend Dependencies
Installs FastAPI, Uvicorn, Scikit-learn, Firebase Admin, and llama-cpp-python CPU/CUDA wheels.

In [ ]:
import sys
import subprocess

print("[1/2] Installing core Python dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
    'fastapi', 'uvicorn', 'numpy', 'pandas', 'scikit-learn', 'firebase-admin', 
    'huggingface-hub', 'diskcache', 'json_repair', 'requests'], check=True)

print("[2/2] Installing llama-cpp-python (optimized CPU wheels)...")
# Uses official pre-compiled wheels for fast installation without local C++ compiler dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cpu',
    'llama-cpp-python'], check=True)

print("\nVerifying installations:")
import fastapi
import uvicorn
import llama_cpp
import firebase_admin
import sklearn
print(f"  - FastAPI          : {fastapi.__version__}")
print(f"  - Uvicorn          : {uvicorn.__version__}")
print(f"  - llama-cpp-python : {llama_cpp.__version__}")
print(f"  - Firebase Admin   : {firebase_admin.__version__}")
print(f"  - Scikit-Learn     : {sklearn.__version__}")
print("\n[DEPENDENCIES READY] All backend packages installed cleanly.")


## Cell 4: Download Gemma 2B GGUF Model
Downloads the instruction-tuned `gemma-2-2b-it-Q4_K_M.gguf` (~1.71 GB) directly into `/content/Detoura/models/` without committing it to Git.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = 'bartowski/gemma-2-2b-it-GGUF'
FILENAME = 'gemma-2-2b-it-Q4_K_M.gguf'
MODELS_DIR = Path('/content/Detoura/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
TARGET_FILE = MODELS_DIR / FILENAME

if TARGET_FILE.exists() and TARGET_FILE.stat().st_size > 1_000_000_000:
    print(f"[FOUND] Gemma 2B model already present at {TARGET_FILE} ({TARGET_FILE.stat().st_size / (1024**3):.2f} GiB).")
else:
    print(f"Downloading {FILENAME} from Hugging Face ({REPO_ID})...")
    print("Hugging Face to Colab transfer typically completes in ~20-30 seconds.")
    downloaded_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        local_dir=str(MODELS_DIR)
    )
    print(f"\n[DOWNLOAD SUCCESSFUL] Model saved to: {downloaded_path}")

print(f"Model File Size: {TARGET_FILE.stat().st_size:,} bytes")
os.environ['GEMMA_MODEL_PATH'] = str(TARGET_FILE)


## Cell 5: Configure Firebase Credentials Securely
Upload your Firebase Admin SDK Service Account JSON key (`credentials/*.json`). Credentials remain private in your Colab runtime and are never printed or committed.

In [ ]:
import os
import sys
import json
from pathlib import Path

CRED_PATH = Path('/content/credentials.json')

if not CRED_PATH.exists():
    print("[ACTION REQUIRED] Please upload your Firebase Service Account JSON key:")
    print("(Found locally in D:\\Detoura\\detoura-ml\\credentials\\detoura-f8a96-*.json)")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname, data in uploaded.items():
            if fname.endswith('.json'):
                with open(CRED_PATH, 'wb') as f:
                    f.write(data)
                print(f"\nSaved credentials to {CRED_PATH}.")
                break
    except Exception as e:
        print(f"Notice: {e}")

if CRED_PATH.exists():
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(CRED_PATH)
    os.environ['FIREBASE_PROJECT_ID'] = 'detoura-f8a96'
    
    # Non-destructive connection test
    if '/content/Detoura' not in sys.path:
        sys.path.insert(0, '/content/Detoura')
    from firebase_client import test_connection
    res = test_connection()
    if res.get('status') == 'connected':
        print(f"\n[FIREBASE CONNECTED] Project: {res.get('project_id')}, Firestore: {res.get('firestore')}")
    else:
        print(f"\n[WARNING] Firebase status: {res}")
else:
    print("\n[NOTICE] No credentials file uploaded. Application will operate with Application Default Credentials (ADC).")


## Cell 6: Start FastAPI Server in Background
Launches the production FastAPI application (`app:app`) listening on `127.0.0.1:8000`.

In [ ]:
import os
import sys
import time
import subprocess
import socket

os.chdir('/content/Detoura')
os.environ['PORT'] = '8000'
os.environ['DETOURA_DB_PATH'] = '/content/Detoura/detoura_master-v2.db'
os.environ['GEMMA_MODEL_PATH'] = '/content/Detoura/models/gemma-2-2b-it-Q4_K_M.gguf'
os.environ['GEMMA_THREADS'] = '4'
os.environ['PYTHONPATH'] = '/content/Detoura'

# Terminate any existing background instance on port 8000
subprocess.run('fuser -k 8000/tcp || true', shell=True, capture_output=True)

log_file = open('/content/fastapi.log', 'w')
cmd = [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000', '--workers', '1']
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file)

print(f"Starting FastAPI / Uvicorn server in background (PID: {server_proc.pid})...")
# Wait up to 30 seconds for the server to bind
started = False
for i in range(30):
    time.sleep(1)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', 8000)) == 0:
            started = True
            break

if started:
    print("[FASTAPI STARTED] Backend is listening on http://127.0.0.1:8000")
else:
    print("[ERROR] Server failed to start within 30 seconds. Inspecting log:")
    with open('/content/fastapi.log') as f:
        print(f.read())


## Cell 7: Verify Local Backend Endpoints
Runs automated tests against the running server for `/health`, `/destinations`, `/recommend`, and `/api/recommendations`.

In [ ]:
import urllib.request
import json

base = 'http://127.0.0.1:8000'

def test_get(endpoint):
    req = urllib.request.Request(base + endpoint, headers={'User-Agent': 'Colab-Verifier'})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.status, json.loads(r.read().decode())

def test_post(endpoint, payload):
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(base + endpoint, data=data, headers={
        'Content-Type': 'application/json',
        'User-Agent': 'Colab-Verifier'
    })
    with urllib.request.urlopen(req, timeout=25) as r:
        return r.status, json.loads(r.read().decode())

print("--- Testing /health ---")
status, res = test_get('/health')
print(f"Status: {status} -> {res}")

print("\n--- Testing /health/firebase ---")
status, res = test_get('/health/firebase')
print(f"Status: {status} -> {res}")

print("\n--- Testing /destinations (SQLite Catalog) ---")
status, res = test_get('/destinations')
print(f"Status: {status} -> Loaded {len(res)} destinations from SQLite catalog.")

print("\n--- Testing /recommend (Mathematical Model 2 Scoring) ---")
payload = {
    'user_prefs': {'nature': 0.85, 'relaxation': 0.90, 'culture': 0.70, 'adventure': 0.65},
    'weights': {'w_sim': 1.0, 'w_season': 1.0, 'w_crowd': 1.0, 'w_gem': 1.0},
    'month': 10
}
status, recs = test_post('/recommend', payload)
print(f"Status: {status} -> Top: {recs[0]['name']} (Score: {recs[0]['detoura_score']:.4f})")

print("\n--- Testing Token Isolation on Protected Endpoints ---")
try:
    urllib.request.urlopen(base + '/api/users/test_user_123/travel-dna')
    print("[FAIL] Protected endpoint was accessible without authorization!")
except urllib.error.HTTPError as e:
    print(f"[PASS] Correctly blocked unauthorized access with HTTP {e.code}: {e.reason}")

print("\n[ALL LOCAL BACKEND VERIFICATIONS PASSED] ✓")


## Cell 8: Start Cloudflare Quick Tunnel
Installs official `cloudflared` binary and creates a secure, free public HTTPS tunnel.

In [ ]:
import subprocess
import time
import re
import urllib.request
from pathlib import Path
from IPython.display import display, HTML

print("[1/2] Installing official cloudflared Linux binary...")
cf_bin = Path('/usr/local/bin/cloudflared')
if not cf_bin.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        str(cf_bin)
    )
    subprocess.run(['chmod', '+x', str(cf_bin)], check=True)
print("cloudflared binary ready.")

print("[2/2] Launching Cloudflare Quick Tunnel forwarding to port 8000...")
subprocess.run('pkill cloudflared || true', shell=True)

cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=cf_log,
    stderr=cf_log
)

tunnel_url = None
for i in range(25):
    time.sleep(1)
    if Path('/content/cloudflared.log').exists():
        with open('/content/cloudflared.log') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*65)
    print("  CLOUDFLARE QUICK TUNNEL LIVE!")
    print(f"  PUBLIC BACKEND HTTPS URL: {tunnel_url}")
    print("="*65)
    display(HTML(f"""
    <div style="padding:16px;background:#064e3b;color:#ecfdf5;border-radius:8px;font-family:sans-serif;">
        <h3 style="margin-top:0;">🚀 Your Public Detoura Backend is Live!</h3>
        <p><b>Public URL:</b> <a href="{tunnel_url}/health" target="_blank" style="color:#6ee7b7;font-weight:bold;">{tunnel_url}</a></p>
        <p>Copy this URL and configure it in your Vercel project as <code>VITE_API_BASE_URL</code>.</p>
    </div>
    """))
    with open('/content/tunnel_url.txt', 'w') as f:
        f.write(tunnel_url)
else:
    print("[ERROR] Tunnel did not return URL within 25 seconds. Cloudflared log:")
    with open('/content/cloudflared.log') as f:
        print(f.read())


## Cell 9: Public Endpoint Verification
Sends public HTTPS requests to the Cloudflare Tunnel URL to verify routing, TLS termination, and CORS.

In [ ]:
import urllib.request
import json
from pathlib import Path

tunnel_url = Path('/content/tunnel_url.txt').read_text().strip()

print(f"Sending public test request to: {tunnel_url}/health")
req = urllib.request.Request(f"{tunnel_url}/health", headers={
    'User-Agent': 'Detoura-Public-Verification',
    'Origin': 'https://detoura.vercel.app'
})

with urllib.request.urlopen(req, timeout=20) as resp:
    print(f"HTTP Status Code: {resp.status}")
    cors_header = resp.headers.get('Access-Control-Allow-Origin')
    print(f"CORS Access-Control-Allow-Origin: {cors_header}")
    body = json.loads(resp.read().decode())
    print(f"Response Body: {body}")

assert resp.status == 200, "Expected status 200"
assert body.get('status') == 'ok', "Expected status: ok"
print("\n[PUBLIC ENDPOINT CONFIRMED] External traffic routes seamlessly to FastAPI!")


## Cell 10: Connect to Vercel Production Frontend
Follow these 3 simple steps to point your production frontend to the live Colab backend:

1. **Copy Your Live Backend URL:**
   * URL: `<TUNNEL_URL>` (from Cell 8 above)

2. **Update Vercel Environment Variables:**
   * Open [Vercel Dashboard](https://vercel.com/) $\rightarrow$ Select project **detoura** $\rightarrow$ **Settings** $\rightarrow$ **Environment Variables**.
   * Add or edit the variable:
     * **Key:** `VITE_API_BASE_URL`
     * **Value:** `<YOUR_TUNNEL_URL>` *(e.g. `https://example.trycloudflare.com` without trailing slash)*
     * **Target:** `Production` and `Preview`

3. **Redeploy on Vercel:**
   * Go to **Deployments** $\rightarrow$ Click the **three dots (...)** next to the latest production deployment $\rightarrow$ Click **Redeploy**.
   * *(Note: Vite inlines `VITE_API_BASE_URL` into JavaScript bundles at build time, so triggering a redeploy is mandatory).*

> **⚠️ Colab & Quick Tunnel Lifespan Note:**
> Free Google Colab sessions remain active while your browser tab is open (up to 12 hours max per session). When Colab restarts, a fresh Quick Tunnel URL is generated, requiring updating `VITE_API_BASE_URL` in Vercel. This provides **100% free hosting with zero charges and zero payment methods**.
